# 00 — The Laplace equation: why every instrument in this series starts with it, and how a computer actually solves it

Every notebook after this one does the same first step: *solve the
field*. Before trusting that step, you should know what equation is
being solved, why it is the right equation, and what the computer is
actually doing while you wait. This notebook builds all three, from
Coulomb's law to a working solver you write yourself in ~20 lines, and
ends by handing off to the industrial version `ion_gym` uses.

**No step is assumed.** If you know Gauss's law and Taylor series, you
have everything required.

## 1 · From charges to a differential equation

Electrostatics starts from two facts:

1. **Gauss's law**: electric field lines begin and end on charge,
   $$\nabla \cdot \vec{E} = \rho / \varepsilon_0,$$
   where $\rho$ is charge density and $\varepsilon_0$ the vacuum
   permittivity. Read $\nabla\cdot\vec{E}$ ("divergence") as *net
   outflow of field lines per unit volume*.

2. **The field is conservative**: pushing a charge around any closed
   loop costs zero net work, so $\vec{E}$ can be written as the slope
   of a single scalar landscape, the **potential** $\varphi$:
   $$\vec{E} = -\nabla \varphi.$$
   The minus sign makes positive charges roll *downhill* in $\varphi$.

Substitute (2) into (1):
$$\nabla \cdot (-\nabla\varphi) = \rho/\varepsilon_0
\quad\Longrightarrow\quad
\nabla^2 \varphi = -\rho/\varepsilon_0.$$

This is **Poisson's equation**. Now the key physical observation for
ion optics: in the vacuum *between* electrodes there is no charge
density worth naming — the ions we fly are so few that their own field
is negligible (a real approximation, called neglecting *space charge*,
and it is stated, not hidden). With $\rho = 0$:

$$\boxed{\ \nabla^2\varphi = 0\ }$$

the **Laplace equation**. Every einzel lens, quadrupole, funnel, drift
tube, and Orbitrap in this series is, mathematically, a boundary-value
problem for this one equation.

## 2 · What the equation *means*: the average property

Write the Laplacian out in 2-D:
$$\nabla^2\varphi = \frac{\partial^2 \varphi}{\partial x^2}
                  + \frac{\partial^2 \varphi}{\partial y^2} = 0.$$

A second derivative measures *curvature* — how much a function at a
point differs from the average of its surroundings. Setting the total
curvature to zero says:

> **The potential at every vacuum point equals the average of the
> potential around it.**

That single sentence carries three consequences you will use all
series:

* **No bumps or dips in vacuum.** A local maximum would be a point
  sitting *above* its average — forbidden. Extremes of $\varphi$ live
  only on the electrodes.
* **Earnshaw's theorem, for free.** Since $\varphi$ has no minimum in
  vacuum, no arrangement of static voltages can trap a charge in all
  three directions at once. This is *why* the Paul trap and Orbitrap
  notebooks need a trick (oscillating fields, or angular momentum) —
  the trick is forced by the mathematics you are looking at.
* **A solving strategy.** If the answer is "every point is the average
  of its neighbours," then *repeatedly replacing every point by that
  average* should walk any starting guess toward the answer. That is
  exactly the algorithm below.

## 3 · Boundary conditions, and why a JSON file can define an instrument

The equation alone has infinitely many solutions ($\varphi = $ const,
$\varphi = x$, $\varphi = xy$, ...). What pins down *the* field of
*your* instrument is the boundary: every metal electrode surface is
held at a known voltage by its power supply (a **Dirichlet** boundary
condition — the value of $\varphi$ is prescribed there).

The **uniqueness theorem** says that once $\varphi$ is fixed on the
whole boundary, the Laplace solution inside is *unique*. This is the
quiet foundation of the entire deck system: a JSON file that records
only *geometry + voltages* is a complete definition of the field — no
further physics input exists to disagree about.

One more gift from the mathematics: the equation is **linear**. If
$\varphi_A$ solves it with electrode A at 1 V (others grounded) and
$\varphi_B$ with electrode B at 1 V, then
$V_A\,\varphi_A + V_B\,\varphi_B$ solves it with the electrodes at
$V_A$ and $V_B$. Solve once per electrode, then *any* voltage setting
is a weighted sum — this is why `ion_gym` re-weights cached basis
solutions >200× faster than re-solving, and why the workflow is
geometry → solve → *then* choose voltages.

## 4 · Discretizing: from calculus to arithmetic

A computer cannot hold a continuous $\varphi(x,y)$; it holds samples
$\varphi_{i,j}$ on a grid with spacing $h$ (the **pitch**, in mm — the
same `mm_per_gu` you will meet in every deck). Derivatives become
differences. Taylor-expand $\varphi$ one grid step each way along $x$:

$$\varphi_{i\pm1,j} = \varphi_{i,j} \pm h\,\varphi_x
   + \tfrac{h^2}{2}\varphi_{xx} \pm \tfrac{h^3}{6}\varphi_{xxx}
   + \mathcal{O}(h^4).$$

Add the two expansions — the odd terms cancel *exactly*:

$$\varphi_{i+1,j} + \varphi_{i-1,j} = 2\varphi_{i,j}
   + h^2 \varphi_{xx} + \mathcal{O}(h^4)
\;\Longrightarrow\;
\varphi_{xx} \approx
   \frac{\varphi_{i+1,j} - 2\varphi_{i,j} + \varphi_{i-1,j}}{h^2}.$$

Do the same in $y$, add, and set the sum to zero (that *is* the Laplace
equation). The $h^2$ cancels, and solving for the centre point gives
the **five-point stencil**:

$$\varphi_{i,j} = \tfrac14\big(\varphi_{i+1,j} + \varphi_{i-1,j}
              + \varphi_{i,j+1} + \varphi_{i,j-1}\big).$$

The continuous "average property" of §2 has become literal arithmetic:
*each vacuum point should equal the mean of its four neighbours*, with
an error that shrinks as $h^2$ (that leftover $\mathcal{O}(h^4)$ term,
divided by $h^2$). Metal points don't relax — their value is the
boundary condition.

In [ ]:
# --- Named parameters (the only cell with tuning choices) --------------
N = 61            # grid points per side for the toy problems; 61 keeps
                  # every toy solve under a second while leaving visible
                  # structure. Odd, so a mid-line exists exactly.
V_TOP = 100.0     # [V] top-plate voltage for the parallel-plate problem;
                  # any value works (linearity!), 100 makes % errors easy
N_SHOW = (10, 100, 1000)   # Jacobi iteration counts to snapshot: chosen a
                  # decade apart so the slow diffusion of information is
                  # visible on one plot
OMEGA = 1.9       # SOR over-relaxation factor; the textbook near-optimal
                  # value for this grid is 2/(1+sin(pi/N)) ~ 1.90
TOL = 1e-6        # [V] max update per sweep at which we call it converged:
                  # 1e-6 of a 100 V problem is far below any physics here
FIG_DPI = 300     # [dots/inch] rendering resolution for every figure in
                  # this notebook (toy panels and the framework render).
                  # 300 is the project default (viz_core.DEFAULT_DPI,
                  # by default); lower it to trade sharpness
                  # against embedded-image size in the saved notebook
EINZEL_DECK = "../examples/einzel_round_r-z.json"
                  # the real instrument used for the closing handoff


## 5 · Your first field solver: parallel plates

The simplest instrument imaginable: a grounded plate at the bottom,
a plate at `V_TOP` on top, vacuum between. The exact answer is known —
with no $x$-dependence anywhere, $\varphi_{yy}=0$, so $\varphi$ rises
*linearly* from 0 to `V_TOP`. Perfect for a first test, because every
digit of error is the algorithm's fault, not the physics'.

The algorithm (**Jacobi relaxation**) is the average property, applied:

1. Set the boundary rows to their voltages. Guess anything inside
   (zeros are fine — uniqueness says the start cannot matter).
2. Replace every interior point by the mean of its four neighbours.
3. Repeat until nothing moves.

Watch *how* it converges: each sweep lets a point feel only its
immediate neighbours, so the top plate's influence walks into the
interior roughly **one grid cell per sweep** — information diffuses.
That is why naive relaxation is slow, and why §7 exists.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def jacobi_sweep(phi, metal):
    '''One Jacobi sweep: every vacuum point -> mean of its 4 neighbours.
    `metal` marks fixed-voltage points; they are simply put back after.'''
    new = phi.copy()
    new[1:-1, 1:-1] = 0.25 * (phi[2:, 1:-1] + phi[:-2, 1:-1]
                              + phi[1:-1, 2:] + phi[1:-1, :-2])
    new[metal] = phi[metal]         # electrodes do not relax
    return new

# plates: bottom row 0 V, top row V_TOP; sides periodic-in-spirit --
# we pin them to the exact linear ramp so the 1-D character is clean
phi = np.zeros((N, N))
metal = np.zeros((N, N), bool)
metal[0, :] = metal[-1, :] = True          # the two plates
phi[-1, :] = V_TOP
ramp = np.linspace(0.0, V_TOP, N)
metal[:, 0] = metal[:, -1] = True          # side rails: exact ramp
phi[:, 0] = phi[:, -1] = ramp

exact = np.tile(ramp[:, None], (1, N))     # the closed-form answer
snaps, history, it = {}, [], 0
while True:
    nxt = jacobi_sweep(phi, metal)
    it += 1
    history.append(np.abs(nxt - phi).max())
    phi = nxt
    if it in N_SHOW:
        snaps[it] = phi.copy()
    if history[-1] < TOL or it >= 20000:
        break
print(f"Jacobi: {it} sweeps to reach a max update < {TOL:g} V; "
      f"final max |phi - exact| = {np.abs(phi - exact).max():.2e} V "
      f"of {V_TOP:g} V")


In [ ]:
# Watch information diffuse: the mid-column profile after 10/100/1000
# sweeps vs the exact ramp, and the update size per sweep (log scale).
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.4), dpi=FIG_DPI)
y = np.arange(N)
for k in N_SHOW:
    axs[0].plot(snaps[k][:, N // 2], y, label=f"{k} sweeps")
axs[0].plot(exact[:, N // 2], y, "k--", lw=1, label="exact (linear)")
axs[0].set_xlabel("phi (V)"); axs[0].set_ylabel("grid row (bottom -> top)")
axs[0].set_title("the answer creeps in ~1 cell/sweep", fontsize=9)
axs[0].legend(fontsize=7)
axs[1].semilogy(history)
axs[1].set_xlabel("sweep"); axs[1].set_ylabel("max update (V)")
axs[1].set_title("convergence is geometric but SLOW", fontsize=9)
fig.suptitle(f"Jacobi relaxation, {N}x{N} plates at {V_TOP:g} V",
             fontsize=10)
fig.tight_layout()


**Read the left panel before moving on.** After 10 sweeps the interior
still barely knows the top plate exists; after 100 the ramp is forming;
after 1000 it is nearly exact. Nothing about the *answer* required
thousands of sweeps — only the *messaging speed* of the algorithm did.
The right panel shows the update size falling geometrically, but with a
ratio painfully close to 1: for an $N \times N$ grid, Jacobi needs
$\mathcal{O}(N^2)$ sweeps. Double the resolution, quadruple the wait —
*before* counting the 4× more points per sweep. Real solves at
instrument resolution would take hours this way. Two classic ideas fix
it.

## 6 · A real 2-D field: one biased electrode in a grounded box

Before speeding anything up, one problem the plates cannot show: put a
short biased electrode segment inside a grounded box and watch the
field *fringe*. There is no closed form here — this is the first time
the solver tells you something you did not already know. The contour
plot is worth staring at: equipotentials leave the electrode parallel
to it, bulge outward, and meet the grounded walls — and every vacuum
point still sits exactly at the average of its neighbours.

In [ ]:
phi2 = np.zeros((N, N))
metal2 = np.zeros((N, N), bool)
metal2[0, :] = metal2[-1, :] = metal2[:, 0] = metal2[:, -1] = True  # box, 0 V
metal2[N // 2, N // 4:N // 2] = True                # a short plate segment
phi2[N // 2, N // 4:N // 2] = V_TOP                 # ... at V_TOP
it2 = 0
while True:
    nxt = jacobi_sweep(phi2, metal2)
    it2 += 1
    d = np.abs(nxt - phi2).max()
    phi2 = nxt
    if d < TOL or it2 >= 40000:
        break
print(f"fringing problem: {it2} Jacobi sweeps")
fig, ax = plt.subplots(figsize=(4.6, 4.2), dpi=FIG_DPI)
cs = ax.contour(phi2, levels=15, linewidths=0.8)
ax.clabel(cs, fontsize=6, fmt="%.0f")
ax.plot(np.arange(N // 4, N // 2), [N // 2] * (N // 2 - N // 4),
        "k-", lw=3, label=f"electrode, {V_TOP:g} V")
ax.set_title("equipotentials fringe around a finite electrode\n"
             "(box walls grounded)", fontsize=9)
ax.legend(fontsize=7); ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()


## 7 · Making it fast: Gauss–Seidel, over-relaxation, red–black

**Idea 1 — use fresh values immediately (Gauss–Seidel).** Jacobi
computes a whole new grid from the old one. If instead you sweep in
place, each update already sees its earlier-updated neighbours —
information now travels *many* cells per sweep. Cost: the loop can no
longer be one vectorized NumPy line, because order matters.

**Idea 2 — overshoot on purpose (SOR).** Each update moves a point
*toward* its neighbour average; since the whole field is still rising
together, the move is systematically too timid. **Successive
over-relaxation** scales the step by $\omega \in (1, 2)$:
$$\varphi \leftarrow \varphi + \omega\,(\text{avg} - \varphi).$$
At the textbook-optimal $\omega \approx 2/(1 + \sin(\pi/N))$ the sweep
count drops from $\mathcal{O}(N^2)$ to $\mathcal{O}(N)$ — for large
grids, orders of magnitude.

**Idea 3 — colour the grid like a chessboard (red–black).** In-place
sweeping seems doomed to run one point at a time. But look at the
stencil: a "red" point's update reads only "black" neighbours and vice
versa. So update *all red at once*, then *all black at once* — two
vectorized (or fully parallel) half-sweeps with Gauss–Seidel's fresh
values. This is exactly the scheme in `ion_gym`'s production solver
(Numba-parallel red–black SOR).

In [ ]:
def redblack_sor_sweep(phi, metal, omega):
    '''One red-black SOR sweep: two vectorized half-sweeps.'''
    for colour in (0, 1):
        avg = np.zeros_like(phi)
        avg[1:-1, 1:-1] = 0.25 * (phi[2:, 1:-1] + phi[:-2, 1:-1]
                                  + phi[1:-1, 2:] + phi[1:-1, :-2])
        ii, jj = np.meshgrid(np.arange(phi.shape[0]),
                             np.arange(phi.shape[1]), indexing="ij")
        upd = ((ii + jj) % 2 == colour) & ~metal
        upd[0, :] = upd[-1, :] = upd[:, 0] = upd[:, -1] = False
        phi[upd] += omega * (avg[upd] - phi[upd])
    return phi

phi3 = np.zeros((N, N)); phi3[-1, :] = V_TOP
phi3[:, 0] = phi3[:, -1] = ramp
it3, hist3 = 0, []
while True:
    before = phi3.copy()
    phi3 = redblack_sor_sweep(phi3, metal, OMEGA)
    it3 += 1
    hist3.append(np.abs(phi3 - before).max())
    if hist3[-1] < TOL or it3 >= 20000:
        break
print(f"same plates problem -- Jacobi took {it} sweeps; "
      f"red-black SOR (omega={OMEGA}) takes {it3} sweeps "
      f"({it/it3:.0f}x fewer); max |phi - exact| = "
      f"{np.abs(phi3 - exact).max():.2e} V")
fig, ax = plt.subplots(figsize=(5.2, 3.0), dpi=FIG_DPI)
ax.semilogy(history, label=f"Jacobi ({it} sweeps)")
ax.semilogy(hist3, label=f"red-black SOR, omega={OMEGA} ({it3} sweeps)")
ax.set_xlabel("sweep"); ax.set_ylabel("max update (V)")
ax.set_title("same equation, same answer, ~two orders less waiting",
             fontsize=9)
ax.legend(fontsize=8); fig.tight_layout()


## 8 · What the pitch buys, and what it costs

The stencil derivation left an $\mathcal{O}(h^2)$ error in the
*interior*: halve the pitch and interior errors drop ~4×. But there is
a second, less forgiving error at *curved electrode surfaces*: a smooth
spindle or ring rasterized onto square cells lands its surface wrong by
up to $h/2$, and that surface-placement error is only
$\mathcal{O}(h)$ — halve the pitch, halve (not quarter) it.

This is not folklore in this repository; it was **measured on a real
instrument this week**. The Orbitrap deck's axial frequency, whose
value is set entirely by the electrode surfaces, came out −0.74 % from
ideal at $h = 0.05$ mm and −0.38 % at $h = 0.025$ mm — the deficit
halved with $h$, the fingerprint of the surface term (evidence:
The solver even prints an advisory naming
this term when curved metal is rasterized). The lesson to carry into
every deck you write: **pitch is a physics choice**. Quote it with any
number you certify, and when a result depends on near-surface fields,
demonstrate an $h$-plateau before believing it.

## 9 · Handoff: the industrial version of what you just built

`ion_gym`'s solver is your §7 code, grown up: red–black SOR compiled by
Numba across cores, on r-z / planar / 3-D grids, with per-electrode
**basis solutions** (§3's linearity) cached so a voltage change is a
re-weight, not a re-solve. Below, the same call every instrument
notebook opens with — on the real einzel lens deck — followed by the
deck's full fan of ions (a disc source, so the rays enter off-axis and
the lens has something to focus), rendered from the solver's own mask
by the framework. From nb01 onward, when the field "just appears," you
now know the two seconds of red–black SOR that put it there.

In [ ]:
import time
from ion_gym.io.sim_spec import SimSpec
from ion_gym.io.deck_params import describe_deck
from ion_gym.physics.sim_build import build_run
from ion_gym.viz.viz_core import scene_from_simspec, render_mpl

spec = SimSpec.from_json(EINZEL_DECK)
describe_deck(spec)          # everything inherited from the deck, stated
t0 = time.time()
model, fly, cols, births = build_run(spec)
print(f"production solve: grid {model.A.shape} at "
      f"{spec.geometry.mm_per_gu} mm/gu in {time.time()-t0:.1f} s "
      f"(cached thereafter)")
# Fly EVERY birth the deck declares. The source is a disc of radius
# source.r_mm normal to the beam axis, so the ions enter as a FAN of
# off-axis rays -- exactly what a lens exists to focus. A single on-axis
# ion would fly straight through any einzel voltage and show nothing.
trajs, fates = [], []
for i in range(len(births)):
    tr, st = fly(i)
    trajs.append(tr)
    fates.append(st["kind"])
r_birth = np.hypot(births[:, 1], births[:, 2])
print(f"flew {len(births)} ions from the deck's disc source: birth radii "
      f"{r_birth.min():.2f}-{r_birth.max():.2f} mm "
      f"(declared r_mm = {spec.source.r_mm:g})")
fig = render_mpl(scene_from_simspec(
    spec, model, field="phi", trajs=trajs, fates=fates,
    title=("The same mathematics at instrument scale -- einzel lens, "
           f"solved potential + {len(births)} flown ions from a "
           f"{spec.source.r_mm:g} mm disc source | pitch "
           f"{spec.geometry.mm_per_gu} mm")), layout="column",
    dpi=FIG_DPI)
fig                          # last expression -> the figure displays


## Read-out

* **§5–6 figures**: relaxation walks any starting guess to the unique
  boundary-determined solution; information moves ~1 cell/sweep under
  Jacobi, which is why the fringing problem needed tens of thousands of
  sweeps. *Falsifying picture*: a converged profile that disagrees with
  the exact linear ramp by more than round-off — that would mean the
  stencil, not the physics, is wrong.
* **§7 figure**: red–black SOR reaches the *same* answer in ~two orders
  of magnitude fewer sweeps. *Falsifying picture*: SOR converging to a
  different field than Jacobi (relaxation schemes change speed, never
  the fixed point), or diverging at $\omega < 2$.
* **§8**: interior error is $\mathcal{O}(h^2)$; curved-surface error is
  $\mathcal{O}(h)$ and was measured halving with pitch on the Orbitrap
  deck. *Falsifying result*: a certified observable that fails to
  plateau as $h$ shrinks.
* **§9 figure**: the production solver is the same algorithm at scale;
  the deck fully determines the field (uniqueness), and voltage changes
  re-weight cached bases (linearity). The fan of off-axis rays bends
  toward the axis through the lens — focusing is the visible signature
  of the solved field acting. *Falsifying picture*: the fan passing
  through parallel and unbent (no field reached the equation of
  motion), or rays deflecting AWAY from the axis at a focusing tune.

**What was established.** $\nabla^2\varphi=0$ + electrode voltages is a
complete, unique definition of an instrument's field; "every point
equals its neighbours' average" is both the meaning of the equation and
the algorithm that solves it; and the two costs a student must respect
are sweeps (fixed by the scheme) and pitch (a physics choice that must
be quoted).